# Review finished runs

Loads the runs of one flow and variant into tables: scores by document, failing checks with the span each one rests on, the units the model found, and the digests. Nothing here calls the model.

Run from `evals/`: `uv run jupyter lab`, then open this file.

In [ ]:
from pathlib import Path
import pandas as pd

from design import review

EVALS = Path.cwd().resolve()
if EVALS.name == 'notebooks':
    EVALS = EVALS.parent
FLOW = EVALS / 'private' / 'runs' / 'asa'
VARIANT = 'baseline'
pd.set_option('display.max_colwidth', 250)
pd.set_option('display.max_rows', 400)
pd.set_option('display.width', 250)
CASES = review.list_cases(FLOW, VARIANT)
CASES

## Scores by document

Metrics as rows, documents as columns; fractions are the share of units that passed, counts are failures (lower is better). The covariates below the metrics put a size next to each score.

In [ ]:
review.scores_table(FLOW, VARIANT)

## What fails most

Across all documents: checks per metric by verdict, sorted by failure share. The top row is the lever to pull first in the skill.

In [ ]:
review.failure_share(FLOW, VARIANT)

## Failing checks for one metric, across documents

Each row is one check: the unit it concerns, the verdict, the judge's reason (at most 30 words, about the document only) and the quote it rests on, which is a byte-exact slice of the document.

In [ ]:
METRIC = 'paragraph-earns-place'
review.checks_table(FLOW, VARIANT, metric=METRIC, verdict=['fail', 'unknown', 'unverified'])

## One document in full

All non-passing checks of one document, then the units the model extracted from it (decisions, components, claims, relational passages), then the document itself with the `[pNN]` labels so a check's unit can be found.

In [ ]:
CASE = CASES[0] if CASES else ''
CASE

In [ ]:
review.checks_table(FLOW, VARIANT, case=CASE, verdict=['fail', 'unknown', 'unverified'])

In [ ]:
review.units_table(FLOW, VARIANT, case=CASE, kinds=('decision', 'component', 'claim', 'relational_passage', 'detailed_part'))

In [ ]:
print(review.digest(FLOW, VARIANT, CASE))

In [ ]:
print(review.document(FLOW, VARIANT, CASE, marked=True))

## Passing checks too

When calibrating a rubric, the passes matter as much as the fails: a check that passes something you would have failed is a wording problem in the rubric. Change `verdict` to `'pass'` or drop the filter.

In [ ]:
review.checks_table(FLOW, VARIANT, case=CASE, metric='paragraph-earns-place', verdict='pass').head(40)